# Q01: Mixture of Experts (MoE)
**Topic:** LLM | **Difficulty:** Hard | **Time:** 15 min
**Sheet:** GrindLLM50

---

## Question
What is Mixture of Experts? How and when is it used?

## Detailed Answer

### What is MoE?
A **Mixture of Experts** replaces the standard FFN layer in a Transformer with multiple specialized "expert" networks and a **gating/router** that selects which experts to activate per token.

### Architecture
```
Token → Router (small network) → Top-K experts selected → Expert outputs weighted sum → Output
```

**Key formula:**
$$y = \sum_{i=1}^{N} g_i(x) \cdot E_i(x)$$
where $g_i(x) = \text{softmax}(\text{Router}(x))_i$ and typically only top-K (K=1 or 2) are nonzero.

### How It Works
1. **Router**: Linear layer that produces scores for each expert: $s = W_r \cdot x$
2. **Top-K selection**: Only activate K experts (sparse computation)
3. **Expert computation**: Only selected experts process the token
4. **Weighted combination**: Output = weighted sum of selected expert outputs

### Key Parameters
| Parameter | Typical Value | Effect |
|-----------|--------------|--------|
| N (total experts) | 8-128 | More = more capacity |
| K (active experts) | 1-2 | More = more compute, better quality |
| Expert capacity | 1.0-1.5× | Controls load balancing |

### Load Balancing
**Problem**: Router might keep selecting the same experts (rich-get-richer).
**Solution**: Auxiliary load-balancing loss:
$$L_{aux} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i \cdot P_i$$
where $f_i$ = fraction of tokens routed to expert $i$ and $P_i$ = mean router probability for expert $i$.

### When to Use MoE
- **Scaling model capacity without proportional compute increase**
- GPT-4 (rumored 8 experts), Mixtral 8×7B, DeepSeek-MoE, Switch Transformer
- 8×7B MoE ≈ performance of dense 70B model but costs ~14B inference FLOPs

### Advantages & Challenges
| Advantages | Challenges |
|------------|------------|
| More parameters, same compute | Load imbalance |
| Better scaling | Higher memory (all experts stored) |
| Specialized experts | Training instability |
| Faster inference per token | Complex distributed training |

## Key Takeaways
- MoE = **conditional computation** — only activating a subset of parameters per token
- Key insight: 8×7B MoE ≈ 70B performance at ~14B compute cost
- **Load balancing** is the main engineering challenge
- Used in Mixtral, GPT-4 (rumored), DeepSeek, Switch Transformer